# Get To Know A Dataset: MassGIS on AWS

This notebook is a guided introduction to [MassGIS on AWS](https://registry.opendata.aws/massgis_on_aws/), a planned collection of Massachusetts geospatial data hosted in Amazon S3. It demonstrates how to discover the bucket, read a small window from a Cloud-Optimized GeoTIFF (COG), and use SAM 3 to generate candidate building detections.

The registry entry and production bucket are still in draft onboarding. The default values below point to a public staging preview so the workflow can be tested now.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The production bucket layout is being finalized during onboarding. MassGIS plans to group objects by product type, vintage or acquisition, project area or UTM zone, and spatial tile. A possible layout is:

```text
s3://<official-massgis-bucket>/
  orthoimagery/
    vintage=2025/zone=19/
  lidar/ept/
    acquisition=<year-or-project>/
  lidar-derived/cog/
    product=<product-name>/vintage=<year>/
  vector/
    layer=<layer-name>/vintage=<date>/
  metadata/
```

The current staging preview uses `COGs_MM_test/cog_quarters/2025_ortho/zone19/`, where the objects are 2025 Zone 19 orthoimagery COGs. The next cells inspect that current test prefix.

In [ ]:
# Bootstrap the notebook environment.
import importlib
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", package_name])
        importlib.invalidate_caches()
        importlib.import_module(import_name)

core_packages = [
    ("boto3", "boto3"),
    ("botocore", "botocore"),
    ("matplotlib", "matplotlib"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("Pillow", "PIL"),
    ("pyproj", "pyproj"),
    ("rasterio", "rasterio"),
    ("shapely", "shapely"),
    ("huggingface_hub", "huggingface_hub"),
]

for package_name, import_name in core_packages:
    ensure_package(package_name, import_name)

print("Core notebook dependencies are ready.")


In [ ]:
# Requirements for the data-access and visualization sections:
# boto3 >= 1.38
# botocore >= 1.38
# matplotlib >= 3.10
# numpy >= 2.0
# pillow >= 10
# rasterio >= 1.4
#
# Requirements for the SAM 3 section:
# torch >= 2.7
# transformers >= 5.0
# accelerate >= 1.0

import json
import os
from contextlib import contextmanager
from pathlib import Path
from urllib.parse import quote

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from botocore import UNSIGNED
from botocore.config import Config
from matplotlib.patches import Rectangle
from PIL import Image
from pyproj import Transformer
from rasterio.enums import Resampling
from rasterio.features import shapes
from rasterio.windows import Window
from rasterio.warp import transform_geom
from shapely.geometry import mapping, shape
from shapely.ops import unary_union

try:
    from shapely.validation import make_valid
except ImportError:
    make_valid = None


### Configure the public S3 dataset

The staging defaults make this draft notebook runnable before the official program bucket exists. Edit the `PRODUCTION` dictionary and set `USE_STAGING=False` when the production bucket is available. `MASSGIS_COG_KEY` and `MASSGIS_RUN_SAM3` are optional environment overrides. Because the dataset is intended to be public and read-only, the client uses unsigned requests.

In [ ]:
USE_STAGING = True
STAGING = {
    'bucket': 's3-massit-massgis-04',
    'region': 'us-east-1',
    'prefix': 'COGs_MM_test/cog_quarters/2025_ortho/zone19/',
}
PRODUCTION = {
    'bucket': '<OFFICIAL_MASSGIS_OPEN_DATA_BUCKET>',
    'region': '<OFFICIAL_BUCKET_REGION>',
    'prefix': '<OFFICIAL_2025_ORTHO_COG_PREFIX>',
}
DATASET = STAGING if USE_STAGING else PRODUCTION
BUCKET = DATASET['bucket']
BUCKET_REGION = DATASET['region']
COG_PREFIX = DATASET['prefix']
COG_KEY_OVERRIDE = os.environ.get('MASSGIS_COG_KEY') or None
REGISTRY_URL = '<AWS_OPEN_DATA_REGISTRY_DATASET_URL>'
RUN_SAM3 = os.environ.get('MASSGIS_RUN_SAM3', '1') == '1'
WINDOW_SIZE = 1008
CONTEXT_SIZE = 4032
MIN_VALID_FRACTION = 0.60
SCORE_THRESHOLD = 0.50
MASK_THRESHOLD = 0.50
MIN_AREA_M2 = 10.0
MAX_AREA_M2 = 100000.0
PROMPTS = ['building', 'roof', 'rooftop']
SELECTED_PROMPT = 'building'
CANDIDATE_PLACES = [
    {'name': 'North End, Boston', 'lon': -71.0551, 'lat': 42.3650},
    {'name': 'Boston', 'lon': -71.0589, 'lat': 42.3601},
    {'name': 'Cambridge', 'lon': -71.1056, 'lat': 42.3736},
    {'name': 'Worcester', 'lon': -71.8023, 'lat': 42.2626},
]
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

s3 = boto3.client(
    's3',
    region_name=BUCKET_REGION,
    config=Config(signature_version=UNSIGNED),
)

print('Bucket:', BUCKET)
print('Region:', BUCKET_REGION)
print('COG prefix:', COG_PREFIX)
print('SAM3 enabled:', RUN_SAM3)

### Inspect prefixes and objects

Listing prefixes is useful for discovery, while listing objects within a narrow prefix avoids asking users to enumerate an entire multi-vintage bucket. The paginator keeps this example valid when the production bucket grows beyond 1,000 objects.

In [ ]:
def list_common_prefixes(prefix=''):
    prefixes = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, Delimiter='/'):
        prefixes.extend(item['Prefix'] for item in page.get('CommonPrefixes', []))
    return prefixes


def list_objects(prefix):
    objects = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        objects.extend(page.get('Contents', []))
    return objects


print('Top-level prefixes:')
for prefix in list_common_prefixes():
    print(' ', prefix)

print()
print('Prefixes below the configured COG prefix:')
for prefix in list_common_prefixes(COG_PREFIX):
    print(' ', prefix)

zone19_objects = [
    item for item in list_objects(COG_PREFIX)
    if item['Key'].lower().endswith(('.tif', '.tiff'))
]
if not zone19_objects:
    raise RuntimeError('No COG objects found under the configured prefix.')

object_table = pd.DataFrame([
    {
        'key': item['Key'].replace(COG_PREFIX, '', 1),
        'size_gib': item['Size'] / 2**30,
        'last_modified': item['LastModified'],
    }
    for item in zone19_objects
]).sort_values('key')
display(object_table)
print('COG objects found:', len(zone19_objects))

preferred = [item['Key'] for item in zone19_objects if item['Key'].endswith('19TCG_1_cog.tif')]
COG_KEY = COG_KEY_OVERRIDE or (preferred[0] if preferred else sorted(item['Key'] for item in zone19_objects)[0])

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats?

The imagery portion uses Cloud-Optimized GeoTIFF (COG), a GeoTIFF organization that lets software request only the byte ranges needed for metadata, overviews, or a small image window. The 2025 aerial imagery contains four bands: red, green, blue, and near infrared. The natural-color view uses bands 1, 2, and 3; band 4 can be used for color-infrared analysis.

The planned LiDAR point-cloud portion uses Entwine Point Tile (EPT), a tiled JSON and binary representation designed for spatial point-cloud access. LiDAR-derived rasters will use COG where appropriate, and vector layers may use documented GIS formats such as GeoPackage, Shapefile, or GeoJSON.

For background, see the [MassGIS 2025 Aerial Imagery documentation](https://www.mass.gov/info-details/massgis-data-2025-aerial-imagery) and the [COG specification](https://github.com/cogeotiff/cog-spec).

In [ ]:
head = s3.head_object(Bucket=BUCKET, Key=COG_KEY)
size_gib = head['ContentLength'] / 2**30
print('Content type:', head.get('ContentType'))
print('Size: {:.1f} GiB'.format(size_gib))
print('Accepts range requests:', head.get('AcceptRanges') == 'bytes')
print('Last modified:', head.get('LastModified'))

### Q: Can you show us an example of loading data from your dataset?

A full 2025 Zone 19 tile is much larger than a normal model input for SAM3. The following cells inspect COG metadata, find a candidate Massachusetts location in the North End by transforming geographic coordinates into the source CRS, and read only a 1,008 by 1,008 pixel RGB window plus a small context window. This is a range-based read; it does not download the entire object.

In [ ]:
COG_ENV = {
    'AWS_NO_SIGN_REQUEST': 'YES',
    'GDAL_DISABLE_READDIR_ON_OPEN': 'EMPTY_DIR',
    'CPL_VSIL_CURL_ALLOWED_EXTENSIONS': '.tif,.tiff',
    'GDAL_HTTP_MULTIRANGE': 'YES',
}


def s3_https_url(bucket, region, key):
    encoded_key = '/'.join(quote(part) for part in key.split('/'))
    return 'https://{}.s3.{}.amazonaws.com/{}'.format(bucket, region, encoded_key)


@contextmanager
def remote_cog(key):
    with rasterio.Env(**COG_ENV):
        with rasterio.open(s3_https_url(BUCKET, BUCKET_REGION, key)) as src:
            yield src


cog_metadata = []
for item in zone19_objects:
    with remote_cog(item['Key']) as src:
        cog_metadata.append({
            'key': item['Key'],
            'width': src.width,
            'height': src.height,
            'count': src.count,
            'dtype': src.dtypes[0],
            'crs': src.crs.to_string(),
            'bounds': tuple(src.bounds),
            'resolution': tuple(src.res),
            'block_shapes': src.block_shapes,
            'overviews': src.overviews(1),
        })

display(pd.DataFrame(cog_metadata)[['key', 'width', 'height', 'count', 'dtype', 'crs', 'resolution', 'overviews']])


def point_in_metadata(place, item):
    transformer = Transformer.from_crs('EPSG:4326', item['crs'], always_xy=True)
    x, y = transformer.transform(place['lon'], place['lat'])
    left, bottom, right, top = item['bounds']
    return left <= x <= right and bottom <= y <= top


candidate_matches = [
    {'place': place, 'metadata': item}
    for place in CANDIDATE_PLACES
    for item in cog_metadata
    if point_in_metadata(place, item)
]
if COG_KEY_OVERRIDE:
    candidate_matches = [match for match in candidate_matches if match['metadata']['key'] == COG_KEY_OVERRIDE]
if not candidate_matches:
    raise RuntimeError('No candidate place was found in the configured COGs. Set MASSGIS_COG_KEY and update CANDIDATE_PLACES for that tile.')
print('Candidate AOIs:', [(match['place']['name'], match['metadata']['key']) for match in candidate_matches])


def window_for_point(src, place, size):
    transformer = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
    x, y = transformer.transform(place['lon'], place['lat'])
    center_col, center_row = (~src.transform) * (x, y)
    col_off = int(round(center_col - size / 2))
    row_off = int(round(center_row - size / 2))
    col_off = min(max(0, col_off), max(0, src.width - size))
    row_off = min(max(0, row_off), max(0, src.height - size))
    return Window(col_off, row_off, min(size, src.width), min(size, src.height))


chip = None
for match in candidate_matches:
    with remote_cog(match['metadata']['key']) as src:
        window = window_for_point(src, match['place'], WINDOW_SIZE)
        rgb_masked = src.read([1, 2, 3], window=window, masked=True)
        rgb_raw = np.ma.filled(rgb_masked, 0)
        valid_mask = np.all(rgb_raw > 0, axis=0)
        valid_mask &= ~np.any(np.ma.getmaskarray(rgb_masked), axis=0)
        if float(valid_mask.mean()) < MIN_VALID_FRACTION:
            continue
        context_window = window_for_point(src, match['place'], CONTEXT_SIZE)
        context_masked = src.read([1, 2, 3], window=context_window, out_shape=(3, 600, 600), resampling=Resampling.average, masked=True)
        context_raw = np.ma.filled(context_masked, 0)
        chip = {
            'place': match['place'], 'key': match['metadata']['key'],
            'window': window, 'context_window': context_window,
            'rgb_raw': rgb_raw, 'valid_mask': valid_mask,
            'context_raw': context_raw,
            'window_transform': src.window_transform(window),
            'source_crs': src.crs, 'source_transform': src.transform,
            'source_width': src.width, 'source_height': src.height,
            'source_band_count': src.count, 'source_dtype': src.dtypes[0],
            'source_res': src.res, 'block_shapes': src.block_shapes,
            'overviews': src.overviews(1),
        }
        break
if chip is None:
    raise RuntimeError('No candidate AOI produced a valid image chip.')

rgb_raw = chip['rgb_raw']
window = chip['window']
window_transform = chip['window_transform']
source_crs = chip['source_crs']
source_width = chip['source_width']
source_height = chip['source_height']
source_band_count = chip['source_band_count']
source_dtype = chip['source_dtype']
window_col = int(window.col_off)
window_row = int(window.row_off)
print('Selected place:', chip['place']['name'])
print('COG:', s3_https_url(BUCKET, BUCKET_REGION, chip['key']))
print('Dimensions:', source_width, 'x', source_height)
print('Bands:', source_band_count, '| dtype:', source_dtype)
print('CRS:', source_crs)
print('Window offset:', window_col, window_row)
print('Valid pixels: {:.1%}'.format(float(chip['valid_mask'].mean())))

In [ ]:
def percentile_stretch(bands, valid_mask, low=2, high=98):
    image = np.moveaxis(bands, 0, -1).astype('float32')
    output = np.zeros_like(image, dtype='float32')
    for band_index in range(image.shape[-1]):
        values = image[:, :, band_index][valid_mask]
        if values.size == 0:
            continue
        lo, hi = np.percentile(values, [low, high])
        if hi <= lo:
            hi = lo + 1
        output[:, :, band_index] = np.clip((image[:, :, band_index] - lo) / (hi - lo), 0, 1)
    return output


valid_mask = chip['valid_mask']
rgb_display = percentile_stretch(rgb_raw, valid_mask)
if source_band_count >= 4:
    with remote_cog(chip['key']) as src:
        cir_masked = src.read([4, 1, 2], window=window, masked=True)
    cir_raw = np.ma.filled(cir_masked, 0)
    cir_display = percentile_stretch(cir_raw, valid_mask)
else:
    cir_display = None

context_valid = np.all(chip['context_raw'] > 0, axis=0)
context_display = percentile_stretch(chip['context_raw'], context_valid)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(rgb_display)
axes[0].set_title('{}: RGB chip'.format(chip['place']['name']))
axes[1].imshow(cir_display if cir_display is not None else rgb_display)
axes[1].set_title('Color infrared (NIR, R, G)' if cir_display is not None else 'CIR unavailable')
axes[0].axis('off')
axes[1].axis('off')
axes[2].imshow(context_display)
context_height, context_width = context_display.shape[:2]
chip_col = (window.col_off - chip['context_window'].col_off) / chip['context_window'].width * context_width
chip_row = (window.row_off - chip['context_window'].row_off) / chip['context_window'].height * context_height
chip_width = window.width / chip['context_window'].width * context_width
chip_height = window.height / chip['context_window'].height * context_height
axes[2].add_patch(Rectangle((chip_col, chip_row), chip_width, chip_height, fill=False, edgecolor='yellow', linewidth=2))
axes[2].set_title('Context and selected chip')
axes[2].axis('off')
fig.suptitle('MassGIS 2025 orthoimagery: {}'.format(chip['key']), y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'massgis_2025_aoi_overview.png', dpi=160, bbox_inches='tight')
plt.show()

### Authenticate with Hugging Face

Run the cell below the first time you use this notebook.

1. Sign in to Hugging Face in your browser.
2. If access to `facebook/sam3` is gated for your account, make sure that request has been approved.
3. Run the login cell and paste your token when prompted.

Do not hard-code your token into the notebook. If you prefer, you can also set `HF_TOKEN` or `HUGGINGFACE_HUB_TOKEN` in your environment before running the cell.

Note: a separate beginner notebook will be made that will not require authentication to HF or access to SAM3 which requires a request to Meta. This notebook may be retained in this state, or edited, to serve as an advanced tutorial.

In [ ]:
# Install the Hugging Face client if needed, then sign in.
!pip -q install -U huggingface_hub

import os
from huggingface_hub import login, notebook_login

# New users: create a token at https://huggingface.co/settings/tokens.
# If you already exported HF_TOKEN or HUGGINGFACE_HUB_TOKEN, the cell will use it.
token = os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACE_HUB_TOKEN')
if token:
    login(token=token)
else:
    notebook_login()


### A small object-detection example with SAM 3

SAM 3 performs promptable concept segmentation: a short text prompt can produce instance masks, bounding boxes, and confidence scores. The model checkpoint is gated on Hugging Face. Before running the next cells, use the Hugging Face authentication cells below, request access to [`facebook/sam3`](https://huggingface.co/facebook/sam3) if you have not already done so, and install the model dependencies. Do not place an access token in this notebook.

The official SAM3 repository currently recommends Python 3.12 or newer, PyTorch 2.7 or newer, and a CUDA-capable GPU for practical inference. A CPU run may be possible but will be slow and memory intensive. Set `MASSGIS_RUN_SAM3=1` in the local environment to load the official `Sam3Model` and `Sam3Processor` Transformers API.

In [ ]:
# Optional SAM 3 dependencies.
if RUN_SAM3:
    import importlib
    import subprocess
    import sys

    def ensure_package(package_name, import_name=None):
        import_name = import_name or package_name
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"Installing {package_name}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", package_name])
            importlib.invalidate_caches()
            importlib.import_module(import_name)

    for package_name, import_name in [
        ("transformers", "transformers"),
        ("accelerate", "accelerate"),
    ]:
        ensure_package(package_name, import_name)

    print("SAM 3 dependencies are ready. PyTorch must already be available in the runtime.")
else:
    print("SAM 3 is disabled. Set MASSGIS_RUN_SAM3=1 to install the optional model dependencies and load the model.")


In [ ]:
model = None
processor = None
device = None

if RUN_SAM3:
    import torch
    from transformers import Sam3Model, Sam3Processor

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Inference device:', device)
    if device == 'cpu':
        print('For an interactive tutorial, a CUDA GPU is strongly recommended.')
    model_kwargs = {'device_map': 'auto'} if device == 'cuda' else {}
    model = Sam3Model.from_pretrained('facebook/sam3', **model_kwargs)
    if device == 'cpu':
        model = model.to(device)
    processor = Sam3Processor.from_pretrained('facebook/sam3')
else:
    print('SAM3 is disabled. Set MASSGIS_RUN_SAM3=1 and rerun this cell to enable inference.')

We will compare the text prompts `building`, `roof`, and `rooftop` on the same North End chip. This is a candidate visible-roof demonstration, not a validated building-footprint extraction method. Roofs, shadows, vegetation, construction, and image seams can all produce false positives or missed detections.

In [ ]:
prompt_results = {}

def postprocess_sam3_result(result):
    result_masks = result['masks'].detach().cpu().numpy()
    if result_masks.ndim == 4:
        result_masks = result_masks[:, 0]
    result_boxes = result['boxes'].detach().cpu().numpy()
    result_scores = result['scores'].detach().cpu().numpy()
    keep = result_scores >= SCORE_THRESHOLD
    return result_masks[keep].astype(bool), result_boxes[keep], result_scores[keep]

if RUN_SAM3:
    pil_image = Image.fromarray((rgb_display * 255).astype('uint8'), mode='RGB')
    for prompt in PROMPTS:
        inputs = processor(images=pil_image, text=prompt, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            outputs = model(**inputs)
        results = processor.post_process_instance_segmentation(
            outputs, threshold=SCORE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
            target_sizes=inputs.get('original_sizes').tolist(),
        )[0]
        prompt_results[prompt] = postprocess_sam3_result(results)

prompt_table = pd.DataFrame([
    {
        'prompt': prompt,
        'candidate_count': len(prompt_results[prompt][2]) if prompt in prompt_results else 0,
        'median_score': float(np.median(prompt_results[prompt][2])) if prompt in prompt_results and len(prompt_results[prompt][2]) else np.nan,
    }
    for prompt in PROMPTS
])
display(prompt_table)

if SELECTED_PROMPT in prompt_results:
    masks, boxes, scores = prompt_results[SELECTED_PROMPT]
else:
    masks = np.empty((0, rgb_display.shape[0], rgb_display.shape[1]), dtype=bool)
    boxes = np.empty((0, 4), dtype='float32')
    scores = np.empty((0,), dtype='float32')
print('Selected prompt:', SELECTED_PROMPT)
print('Candidate masks:', len(scores))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb_display)
for index, (mask, box, score) in enumerate(zip(masks, boxes, scores), start=1):
    color = plt.cm.tab20((index - 1) % 20)
    overlay = np.zeros((mask.shape[0], mask.shape[1], 4), dtype='float32')
    overlay[:, :, :3] = color[:3]
    overlay[:, :, 3] = mask.astype('float32') * 0.35
    ax.imshow(overlay)
    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.5))
    ax.text(x1, max(0, y1 - 4), '{}: {:.2f}'.format(index, score), color='white', fontsize=8, backgroundcolor='black')
ax.set_title('SAM3 candidate visible roofs: {}'.format(SELECTED_PROMPT))
ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'massgis_2025_sam3_candidates.png', dpi=160, bbox_inches='tight')
plt.show()

### Q: What is one question that you have answered using these data?

For one small orthoimagery window, which image regions look like buildings to a text-prompted open-vocabulary segmentation model? The output is a set of candidate instance masks and pixel-coordinate bounding boxes. Because the source raster has a geospatial transform, those masks can also be exported as geographic features.

In [ ]:
if not source_crs.is_projected:
    raise RuntimeError('The source CRS is not projected; cannot safely calculate square-metre areas.')
pixel_area_m2 = abs(window_transform.a * window_transform.e - window_transform.b * window_transform.d)
valid_chip_area_m2 = float(chip['valid_mask'].sum() * pixel_area_m2)
features = []
for index, (mask, score) in enumerate(zip(masks, scores), start=1):
    pieces = [
        shape(geometry)
        for geometry, value in shapes(mask.astype('uint8'), mask=mask, transform=window_transform)
        if value == 1
    ]
    if not pieces:
        continue
    geometry = unary_union(pieces)
    if not geometry.is_valid and make_valid is not None:
        geometry = make_valid(geometry)
    if geometry.is_empty:
        continue
    area_m2 = float(geometry.area)
    if area_m2 < MIN_AREA_M2 or area_m2 > MAX_AREA_M2:
        continue
    features.append({
        'type': 'Feature',
        'geometry': transform_geom(source_crs, 'EPSG:4326', mapping(geometry)),
        'properties': {
            'candidate_id': index,
            'prompt': SELECTED_PROMPT,
            'confidence': float(score),
            'area_m2': area_m2,
            'perimeter_m': float(geometry.length),
            'source_key': chip['key'],
            'image_year': 2025,
            'method': 'SAM3 candidate visible-roof mask',
        },
    })

geojson = {'type': 'FeatureCollection', 'features': features}
geojson_path = OUTPUT_DIR / 'massgis_2025_candidate_roofs.geojson'
with geojson_path.open('w', encoding='utf-8') as output_file:
    json.dump(geojson, output_file, indent=2)
areas = [feature['properties']['area_m2'] for feature in features]
summary = {
    'candidate_roof_count': len(features),
    'total_candidate_roof_area_m2': float(sum(areas)),
    'median_candidate_roof_area_m2': float(np.median(areas)) if areas else np.nan,
    'valid_chip_area_m2': valid_chip_area_m2,
    'candidate_roof_coverage_percent': float(sum(areas) / valid_chip_area_m2 * 100) if valid_chip_area_m2 else np.nan,
}
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
print('Wrote {} geographic candidate roof features to {}'.format(len(features), geojson_path))
print('Coordinates were transformed to EPSG:4326 for broad GIS compatibility.')

In [ ]:
# Cumulative GIS analysis: aggregate the candidate roof detections from the North End chip into a projected grid.
import math
from shapely.geometry import box

if not features:
    print("No candidate roof features were generated, so there is nothing to aggregate yet.")
else:
    with geojson_path.open('r', encoding='utf-8') as input_file:
        roof_geojson = json.load(input_file)

    projected_crs = 'EPSG:3857'
    projected_roofs = []
    for feature in roof_geojson['features']:
        geometry_3857 = shape(transform_geom('EPSG:4326', projected_crs, feature['geometry']))
        if geometry_3857.is_empty:
            continue
        projected_roofs.append({
            'geometry': geometry_3857,
            'candidate_id': feature['properties']['candidate_id'],
            'confidence': float(feature['properties']['confidence']),
            'area_m2': float(feature['properties']['area_m2']),
        })

    if not projected_roofs:
        print("No valid projected geometries were available for analysis.")
    else:
        candidate_union_3857 = unary_union([item['geometry'] for item in projected_roofs])
        xmin, ymin, xmax, ymax = candidate_union_3857.bounds

        grid_size_m = 500
        pad_m = 750
        xmin = grid_size_m * math.floor((xmin - pad_m) / grid_size_m)
        ymin = grid_size_m * math.floor((ymin - pad_m) / grid_size_m)
        xmax = grid_size_m * math.ceil((xmax + pad_m) / grid_size_m)
        ymax = grid_size_m * math.ceil((ymax + pad_m) / grid_size_m)

        grid_rows = []
        for x in np.arange(xmin, xmax, grid_size_m):
            for y in np.arange(ymin, ymax, grid_size_m):
                cell = box(x, y, x + grid_size_m, y + grid_size_m)
                hits = [item for item in projected_roofs if item['geometry'].intersects(cell)]
                if not hits:
                    continue
                roof_area_m2 = float(sum(item['geometry'].intersection(cell).area for item in hits))
                roof_count = int(len(hits))
                grid_rows.append({
                    'cell_west_m': float(x),
                    'cell_south_m': float(y),
                    'cell_east_m': float(x + grid_size_m),
                    'cell_north_m': float(y + grid_size_m),
                    'roof_count': roof_count,
                    'roof_area_m2': roof_area_m2,
                    'roof_density_per_sq_km': float(roof_count / (grid_size_m * grid_size_m / 1_000_000)),
                })

        grid_df = pd.DataFrame(grid_rows).sort_values(
            ['roof_area_m2', 'roof_count'],
            ascending=False
        ).reset_index(drop=True)

        analysis_summary = pd.DataFrame([{
            'candidate_roof_count': len(projected_roofs),
            'total_candidate_roof_area_m2': float(sum(item['area_m2'] for item in projected_roofs)),
            'analysis_extent_width_km': float((xmax - xmin) / 1000),
            'analysis_extent_height_km': float((ymax - ymin) / 1000),
            'grid_cell_size_m': grid_size_m,
            'nonempty_grid_cells': int(len(grid_df)),
        }])

        display(analysis_summary)
        display(grid_df.head(10))
        print(
            "Prepared a projected analysis grid for the next map view. "
            "The densest cells highlight where candidate roofs cluster spatially."
        )

In [ ]:
# Map the analysis results: draw the projected grid, candidate roof outlines, and the densest cells.
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

def draw_geometry_outline(ax, geometry, **plot_kwargs):
    if geometry.is_empty:
        return
    geom_type = geometry.geom_type
    if geom_type == 'Polygon':
        x, y = geometry.exterior.xy
        ax.plot(x, y, **plot_kwargs)
        for interior in geometry.interiors:
            ix, iy = interior.xy
            ax.plot(ix, iy, **plot_kwargs)
    elif geom_type in ('MultiPolygon', 'GeometryCollection'):
        for part in geometry.geoms:
            draw_geometry_outline(ax, part, **plot_kwargs)

if 'grid_df' not in globals() or 'projected_roofs' not in globals() or not len(projected_roofs):
    print("Run the previous analysis cell first so the map can use the projected grid results.")
else:
    fig, ax = plt.subplots(figsize=(12, 12))

    patches = []
    values = []
    for _, row in grid_df.iterrows():
        patches.append(
            MplPolygon([
                (row['cell_west_m'], row['cell_south_m']),
                (row['cell_east_m'], row['cell_south_m']),
                (row['cell_east_m'], row['cell_north_m']),
                (row['cell_west_m'], row['cell_north_m']),
            ], closed=True)
        )
        values.append(row['roof_area_m2'])

    collection = PatchCollection(
        patches,
        cmap='YlOrRd',
        alpha=0.45,
        linewidth=0.6,
        edgecolor='none'
    )
    collection.set_array(np.asarray(values, dtype='float32'))
    if values:
        collection.set_clim(vmin=0, vmax=max(values))
    ax.add_collection(collection)

    for item in projected_roofs:
        draw_geometry_outline(ax, item['geometry'], color='black', linewidth=0.9, alpha=0.7)

    top_cells = grid_df.head(3)
    ax.scatter(
        top_cells['cell_west_m'] + grid_size_m / 2,
        top_cells['cell_south_m'] + grid_size_m / 2,
        s=120,
        marker='*',
        color='blue',
        edgecolor='white',
        linewidth=0.8,
        label='Top analysis cells'
    )

    ax.set_title('MassGIS candidate roof density map: projected grid over SAM 3 detections')
    ax.set_xlabel('Web Mercator Easting (m)')
    ax.set_ylabel('Web Mercator Northing (m)')
    ax.set_aspect('equal', adjustable='box')

    if values:
        cbar = fig.colorbar(collection, ax=ax, shrink=0.82)
        cbar.set_label('Candidate roof area per grid cell (m²)')

    ax.legend(loc='upper right')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'massgis_2025_roof_density_map.png', dpi=160, bbox_inches='tight')
    plt.show()

    print(
        "The map turns the chip-level detections into a broader spatial pattern: "
        "candidate roofs, local density, and the strongest cells in one view."
    )

### Q: What is one unanswered question that you think could be answered using these data?

How accurately can building footprints be detected across Massachusetts, across imagery vintages, and across different roof, vegetation, and urban-density conditions? A useful follow-up would tile complete COGs into overlapping 1,008-pixel windows, compare SAM 3 masks with a MassGIS building-structure layer or another reference dataset, and report precision, recall, and positional error by region.

When scaling this experiment, keep each tile's CRS and transform, use overlap to reduce edge effects, deduplicate detections across neighboring windows, and validate the confidence threshold separately for each imagery vintage. The 2025 imagery includes a near-infrared band, so comparing RGB prompts with color-infrared visualizations is another promising investigation.

### References

- [MassGIS Data: 2025 Aerial Imagery](https://www.mass.gov/info-details/massgis-data-2025-aerial-imagery)
- [Registry of Open Data on AWS](https://github.com/awslabs/open-data-registry)
- [SAM 3 official repository](https://github.com/facebookresearch/sam3)
- [SAM 3 Transformers documentation](https://huggingface.co/docs/transformers/model_doc/sam3)